### Part 1. Jupyter Notebook for Data Exploration

In [2]:
# Import libraries
import geopandas as gpd
import pandas as pd
import requests
import zipfile
from pathlib import Path

 #### Download and Extract Data

In [3]:
# Define paths and URL
data_url = "https://files.geo.so.ch/ch.so.agi.av.mopublic/aktuell/2601.ch.so.agi.av.mopublic.shp.zip"
raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")
shapefile_zip = raw_data_path / "solothurn.shp.zip"

# Create directories if they don't exist
raw_data_path.mkdir(parents=True, exist_ok=True)
processed_data_path.mkdir(parents=True, exist_ok=True)

# Download the data
response = requests.get(data_url)
with open(shapefile_zip, 'wb') as f:
    f.write(response.content)

# Unzip the file
with zipfile.ZipFile(shapefile_zip, 'r') as zip_ref:
    zip_ref.extractall(raw_data_path)

print("Data downloaded and extracted.")

Data downloaded and extracted.


#### Load and Process Data

In [11]:
# Load the shapefile
shp_path_1 = raw_data_path / "bodenbedeckung.shp"
shp_path_2 = raw_data_path / "bodenbedeckung_proj.shp"

gdf1 = gpd.read_file(shp_path_1)
gdf2 = gpd.read_file(shp_path_2)

# Combine them into one DataFrame using concat
# ignoring index ensures we get a clean 0..n index
gdf = pd.concat([gdf1, gdf2], ignore_index=True)

# Filter the dataset
# 1. egid is not null (NaN)
# 2. art_txt is 'Gebaeude' (Building)
gdf_filtered = gdf[(gdf['egid'].notna()) & (gdf['art_txt'] == 'Gebaeude')].copy()

# Reproject to WGS84 (EPSG:4326)
gdf_wgs84 = gdf_filtered.to_crs(epsg=4326)

print(f"Loaded {len(gdf)} records.")
print(f"filtered down to {len(gdf_wgs84)} records.")
gdf_wgs84.head()

Loaded 11357 records.
filtered down to 4607 records.


,t_id,t_ili_tid,art_txt,bfs_nr,egid,imprtdtum,nachhrung,geometry
11,15,7aa993b9-33bd-408b-817e-85a8e6476955,Gebaeude,2601,191090670.0,2025-12-05,2010-10-18,"POLYGON ((7.52232 47.21152, 7.52232 47.21157, ..."
12,16,e7b11252-4f24-4db8-a435-b360cfceb51f,Gebaeude,2601,1764522.0,2025-12-05,2017-09-19,"POLYGON ((7.52221 47.21161, 7.52222 47.2117, 7..."
13,17,e736d07c-139f-43b9-a254-17f67606db0b,Gebaeude,2601,502361678.0,2025-12-05,2010-10-18,"POLYGON ((7.52216 47.21171, 7.52216 47.21174, ..."
17,21,1e087c22-4981-421a-bb4b-80869f1db9a6,Gebaeude,2601,502361591.0,2025-12-05,2010-08-23,"POLYGON ((7.53491 47.21724, 7.53496 47.21724, ..."
18,22,b5a37152-9c0b-40db-b9d7-3a031430776c,Gebaeude,2601,1766130.0,2025-12-05,2010-08-23,"POLYGON ((7.54488 47.20563, 7.54489 47.20574, ..."


#### Save file as GeoJSON

In [12]:
# Save the final data as GeoJSON
output_path = processed_data_path / "building.geojson"
gdf_wgs84.to_file(output_path, driver='GeoJSON')
print(f"Data saved to {output_path}")

Data saved to ../data/processed/building.geojson
